<a href="https://colab.research.google.com/github/AnshuChedge/Cricket-Athlete-Wikipedia-Scraper/blob/main/TASK_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Cricket Athlete Wikipedia Scraper
===================================
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# ── Configuration ──────────────────────────────────────────────────
HEADERS = {
    "User-Agent": "CricketScraper/1.0 (educational; contact: you@email.com)"
}

PLAYERS = [
    "Virat Kohli",
    "Rohit Sharma",
    "MS Dhoni",
    "Sachin Tendulkar",
    "Jasprit Bumrah",
    "Shubman Gill",
    "Hardik Pandya",
    "Ravindra Jadeja",
    "KL Rahul",
    "Suryakumar Yadav",
]

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"

# ── Step 1: Fetch raw wikitext via Wikipedia API ───────────────────
def fetch_wikitext(player_name):

    params = {
        "action":  "query",
        "format":  "json",
        "titles":  player_name,
        "prop":    "revisions",
        "rvprop":  "content",
        "rvslots": "main",
        "redirects": 1,
    }
    try:
        response = requests.get(
            WIKIPEDIA_API, params=params,
            headers=HEADERS, timeout=15
        )
        response.raise_for_status()
        data  = response.json()
        pages = data["query"]["pages"]
        page  = next(iter(pages.values()))

        if "revisions" not in page:
            print(f"  Warning: No content found for '{player_name}'")
            return None

        return page["revisions"][0]["slots"]["main"]["*"]

    except requests.exceptions.RequestException as e:
        print(f"  Network error for '{player_name}': {e}")
        return None
    except (KeyError, StopIteration) as e:
        print(f"  Parse error for '{player_name}': {e}")
        return None


# ── Step 2: Parse infobox from wikitext ───────────────────────────
def parse_infobox(wikitext):
    """
    Extracts key fields from the Wikipedia infobox using regex.
    The infobox is a template like {{Infobox cricketer | name = ... }}
    """
    if not wikitext:
        return {}

    record = {}

    # Map: wikitext field name → our column name
    field_map = {
        r"name":            "Full Name",
        r"birth_date":      "Date of Birth",
        r"birth_place":     "Birth Place",
        r"country":         "Nationality",
        r"role":            "Role",
        r"batting":         "Batting Style",
        r"bowling":         "Bowling Style",
        r"testdebutdate":   "Test Debut Date",
        r"testdebutagainst":"Test Debut Against",
        r"odidebutdate":    "ODI Debut Date",
        r"odidebutagainst": "ODI Debut Against",
        r"t20idebutdate":   "T20I Debut Date",
        r"t20idebutagainst":"T20I Debut Against",
    }

    for wiki_field, col_name in field_map.items():
        # Match  | field = value  (value ends at next | or })
        pattern = rf"\|\s*{wiki_field}\s*=\s*([^\|\}}]+)"
        match   = re.search(pattern, wikitext, re.IGNORECASE)
        if match:
            raw = match.group(1)
            # Remove [[link|text]] → text,  [[link]] → link
            raw = re.sub(r'\[\[(?:[^\|\]]+\|)?([^\]]+)\]\]', r'\1', raw)
            # Remove templates {{...}}
            raw = re.sub(r'\{\{[^}]*\}\}', '', raw)
            # Remove HTML tags
            raw = re.sub(r'<[^>]+>', '', raw)
            # Remove citation markers [1], [2], etc.
            raw = re.sub(r'\[\d+\]', '', raw)
            val = ' '.join(raw.split()).strip(" |,\n")
            if val:
                record[col_name] = val

    return record


# ── Step 3: Fetch and parse career stats table ─────────────────────
def fetch_career_stats(player_name):
    """
    Uses Wikipedia's parse API to get rendered HTML, then
    uses pandas.read_html() to extract career statistics tables.
    """
    stats = {
        "Matches (Test)":   "-", "Runs (Test)":   "-",
        "Average (Test)":   "-", "Centuries (Test)": "-",
        "Matches (ODI)":    "-", "Runs (ODI)":    "-",
        "Average (ODI)":    "-", "Centuries (ODI)":  "-",
        "Matches (T20I)":   "-", "Runs (T20I)":   "-",
        "Average (T20I)":   "-", "Centuries (T20I)": "-",
        "Wickets (Test)":   "-", "Wickets (ODI)": "-",
        "Wickets (T20I)":   "-",
    }

    # Get rendered HTML of the page
    params = {
        "action": "parse",
        "format": "json",
        "page":   player_name,
        "prop":   "text",
        "redirects": 1,
    }
    try:
        response = requests.get(
            WIKIPEDIA_API, params=params,
            headers=HEADERS, timeout=15
        )
        response.raise_for_status()
        html = response.json()["parse"]["text"]["*"]
        tables = pd.read_html(html)
    except Exception as e:
        print(f"  Stats error for '{player_name}': {e}")
        return stats

    # Keywords to identify format rows and stat columns
    format_keywords = {
        "Test":  ["test"],
        "ODI":   ["odi", "one day", "one-day"],
        "T20I":  ["t20i", "t20 int", "twenty20 int"],
    }
    stat_keywords = {
        "Matches":   ["mat", "m"],
        "Runs":      ["run", "r"],
        "Average":   ["ave", "avg", "ba"],
        "Centuries": ["100", "ton", "cent"],
        "Wickets":   ["wkt", "wkts", "w"],
    }

    for table in tables:
        if table.shape[1] < 4:
            continue

        # Normalize column names to lowercase
        table.columns = [str(c).lower().strip() for c in table.columns]
        first_col = table.columns[0]
        row_labels = table[first_col].astype(str).str.lower().tolist()

        for fmt, fkws in format_keywords.items():
            # Find the row index for this format
            row_idx = next(
                (i for i, v in enumerate(row_labels)
                 if any(kw in v for kw in fkws)),
                None
            )
            if row_idx is None:
                continue

            row_data = table.iloc[row_idx]

            for stat, skws in stat_keywords.items():
                col_key = f"{stat} ({fmt})"
                if stats[col_key] != "-":
                    continue  # already found
                for col in table.columns:
                    if any(kw in col for kw in skws):
                        val = str(row_data.get(col, "")).strip()
                        if val and val.lower() not in ["nan", "-", "–", ""]:
                            stats[col_key] = val
                        break

    return stats


# ── Step 4: Main pipeline ──────────────────────────────────────────
def run_scraper():
    print("\n" + "=" * 58)
    print("   Cricket Athlete Wikipedia Scraper")
    print("=" * 58)

    all_records = []

    for i, player in enumerate(PLAYERS, 1):
        print(f"\n[{i}/{len(PLAYERS)}] {player}")
        print(f"  Fetching infobox  ...", end=" ", flush=True)
        wikitext = fetch_wikitext(player)
        info     = parse_infobox(wikitext)
        print("Done" if info else "Empty")

        print(f"  Fetching stats    ...", end=" ", flush=True)
        stats = fetch_career_stats(player)
        print("Done")

        record = {
            "Player":        player,
            "Wikipedia URL": f"https://en.wikipedia.org/wiki/{player.replace(' ', '_')}",
        }
        record.update(info)
        record.update(stats)
        all_records.append(record)

        time.sleep(1.5)  # polite delay — don't flood Wikipedia

    # ── Build DataFrame with ordered columns ──
    col_order = [
        "Player", "Full Name", "Date of Birth", "Birth Place", "Nationality",
        "Role", "Batting Style", "Bowling Style",
        "Test Debut Date", "Test Debut Against",
        "ODI Debut Date",  "ODI Debut Against",
        "T20I Debut Date", "T20I Debut Against",
        "Matches (Test)",  "Runs (Test)",  "Average (Test)",
        "Centuries (Test)","Wickets (Test)",
        "Matches (ODI)",   "Runs (ODI)",   "Average (ODI)",
        "Centuries (ODI)", "Wickets (ODI)",
        "Matches (T20I)",  "Runs (T20I)",  "Average (T20I)",
        "Centuries (T20I)","Wickets (T20I)",
        "Wikipedia URL",
    ]
    df = pd.DataFrame(all_records)
    ordered = [c for c in col_order if c in df.columns]
    df = df[ordered]

    # ── Save output ──
    output_file = "cricket_athletes_dataset.csv"
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 58)
    print(f"  Saved  →  {output_file}")
    print(f"  Players scraped : {len(df)}")
    print(f"  Total columns   : {len(df.columns)}")
    print("=" * 58)

    # ── Quick preview ──
    preview_cols = [
        "Player", "Role", "Batting Style",
        "Runs (Test)", "Runs (ODI)", "Average (ODI)"
    ]
    preview_cols = [c for c in preview_cols if c in df.columns]
    print("\nPreview (key columns):\n")
    pd.set_option("display.max_colwidth", 22)
    pd.set_option("display.width", 120)
    print(df[preview_cols].to_string(index=False))
    print()

    return df


if __name__ == "__main__":
    df = run_scraper()


   Cricket Athlete Wikipedia Scraper

[1/10] Virat Kohli
  Fetching infobox  ... Done
  Fetching stats    ... 

/tmp/ipykernel_6718/2173823383.py:160: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Done

[2/10] Rohit Sharma
  Fetching infobox  ... Done
  Fetching stats    ... 

/tmp/ipykernel_6718/2173823383.py:160: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Done

[3/10] MS Dhoni
  Fetching infobox  ... Done
  Fetching stats    ... 

...

Done

[9/10] KL Rahul
  Fetching infobox  ... Done
  Fetching stats    ... 

/tmp/ipykernel_6718/2173823383.py:160: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Done

[10/10] Suryakumar Yadav
  Fetching infobox  ... Done
  Fetching stats    ... 

/tmp/ipykernel_6718/2173823383.py:160: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Done

  Saved  →  cricket_athletes_dataset.csv
  Players scraped : 10
  Total columns   : 30

Preview (key columns):

          Player                                Role Batting Style Runs (Test) Runs (ODI) Average (ODI)
     Virat Kohli [[Batting order (cricket)#Top order  Right-handed           -          -             -
    Rohit Sharma [[Batting order (cricket)#Top order  Right-handed           -          -             -
        MS Dhoni                Wicket-keeper-batter  Right-handed          15          -             -
Sachin Tendulkar                    Top order Batter  Right-handed          12   ODI[415]             -
  Jasprit Bumrah                 [[Bowling (cricket)  Right-handed           -          -             -
    Shubman Gill [[Batting order (cricket)#Top order  Right-handed           -          -             -
   Hardik Pandya                         All-rounder  Right-handed           -          -             -
 Ravindra Jadeja                         All-round